Paths (change the wav. file)
- HC_DIR
- PT_DIR

RESULTS_PATH (whatever name you select - does not matter)
- summary_path
- pred_path

Functions 
- scan_audio_folder()
- load_audio()
- redict_probs() (if switching to regression, rename to predict_values())
- AudioDepressionDataset
- DAMLikeModel (mainly the output head if moving to regression)

Variables / Parameters (depends on structure of dataset/extent of fine-tuning)
- MAX_SECONDS
- BATCH_SIZE
- TEST_SIZE
- LEARNING_RATE (possibly retune)
- loss_fn
- groups
- labels
- paths
- train_idx, test_idx (if using official RADAR split instead of GroupShuffleSplit)

DataFrame columns (depends on dataset structure)
- participant_id
- subgroup
- depressed
- file_path
- file_stem

Output filenames (does not matter)
- "dam_whisper_hc_pt_summary.csv"
- "dam_whisper_hc_pt_test_predictions.csv"

Print statements (does not matter)
- HC dir
- PT dir
- Labels — depressed=0 (HC)...
- "DAM-like Whisper (HC/PT)"
- "P(depressed)"
- Metrics

If using binary depression:

- accuracy_score
- f1_score
- roc_auc_score

If using PHQ-8 regression, replace with:

- mean_absolute_error
- mean_squared_error
- r2_score
- Pearson/Spearman correlation (optional)
- Data loading logic

These are probably the biggest changes:

- HC/PT folder scan
- Participant ID extraction from filename
- Label assignment (depressed)
- Grouped train/test split
- Metadata merge between audio files and the RADAR labels CSV

In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    """Resolve repo root whether the notebook cwd is project root or notebooks/Models/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/raw/")


PROJECT = find_project_root()
RAW_ROOT = PROJECT / "data" / "raw"

# Androids-specific folder structure.
# radar_audio: replace HC/PT subfolders with the RADAR raw-audio location.
# Example:
# RADAR_AUDIO_DIR = RAW_ROOT / "radar_audio"
HC_DIR = RAW_ROOT / "HC"   # healthy controls -> label 0  # radar_audio
PT_DIR = RAW_ROOT / "PT"   # patients -> label 1          # radar_audio


def scan_audio_folder(folder: Path, subgroup: str, label: int) -> list[dict]:
    # radar_audio: RADAR may not be split into HC/PT folders.
    # You will probably need a function that scans all .wav files and merges them with a labels/metadata CSV containing PHQ-8/depression labels.

    if not folder.is_dir():
        raise FileNotFoundError(f"Missing audio folder: {folder}")

    rows = []
    for wav in sorted(folder.glob("*.wav")):
        rows.append(
            {
                "file_path": str(wav.resolve()),
                "file": wav.name,
                "file_stem": wav.stem,
                "subgroup": subgroup,      # radar_audio: replace with site/task/language if useful
                "depressed": label,        # radar_audio: derive from PHQ-8 or existing binary label
            }
        )
    return rows


# Androids-specific HC/PT scan.
# radar_audio: replace this with RADAR metadata/audio merge.
hc_rows = scan_audio_folder(HC_DIR, "HC", 0)  # radar_audio
pt_rows = scan_audio_folder(PT_DIR, "PT", 1)  # radar_audio
audio_df = pd.DataFrame(hc_rows + pt_rows)    # radar_audio

# Androids filename-specific participant extraction.
# First token in filenames like 01_CF56_1 -> participant id for grouped splits later.
# radar_audio: replace with RADAR participant_id from metadata if available.
audio_df["participant_id"] = audio_df["file_stem"].str.split("_").str[0]  # radar_audio

# Androids-specific subgroup counts.
# radar_audio: replace with RADAR label/site/task/language summaries.
n_hc = int((audio_df["subgroup"] == "HC").sum())  # radar_audio
n_pt = int((audio_df["subgroup"] == "PT").sum())  # radar_audio

print(f"Project root: {PROJECT}")
print(f"HC dir: {HC_DIR} ({n_hc} .wav)")  # radar_audio
print(f"PT dir: {PT_DIR} ({n_pt} .wav)")  # radar_audio
print(f"Total recordings: {len(audio_df)}")
print(f"Labels — depressed=0 (HC): {(audio_df['depressed'] == 0).sum()} | depressed=1 (PT): {(audio_df['depressed'] == 1).sum()}")  # radar_audio
print(f"Unique participants: {audio_df['participant_id'].nunique()}")

audio_df.head()

In [ ]:
import gc
from functools import partial

import librosa
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperModel, WhisperProcessor

try:
    audio_df
except NameError as e:
    raise RuntimeError("Run the data-scan cell first (audio_df).") from e


# --------------------------------------------------
# Config
# --------------------------------------------------

WHISPER_ID = "openai/whisper-small.en"

TARGET_SR = 16_000
MAX_SECONDS = 30  # radar_audio: may need changing if RADAR clips are longer/shorter

EPOCHS = 8
BATCH_SIZE = 2    # radar_audio: may need tuning depending on RADAR sample size/GPU memory
LEARNING_RATE = 1e-3
TEST_SIZE = 0.2   # radar_audio: may use official train/test split instead
RANDOM_STATE = 42
FREEZE_ENCODER = True
FORCE_CPU = False

# Androids/output-name specific.
# radar_audio: change folder to something like "radar_audio" or "dam_whisper_radar".
RESULTS_PATH = PROJECT / "results" / "metrics" / "kintsugi_health"#/RADAR
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE} | recordings: {len(audio_df)} | freeze encoder: {FREEZE_ENCODER}")


# --------------------------------------------------
# Preprocessing
# --------------------------------------------------

def load_audio(path) -> np.ndarray:
    wav, _ = librosa.load(str(path), sr=TARGET_SR, mono=True)
    max_len = TARGET_SR * MAX_SECONDS
    if len(wav) > max_len:
        wav = wav[:max_len]  # radar_audio: consider random cropping or chunking for longer recordings
    return wav.astype(np.float32)


processor = WhisperProcessor.from_pretrained(WHISPER_ID)


# --------------------------------------------------
# Model (DAM-style: Whisper encoder + MLP head)
# --------------------------------------------------

class DAMLikeModel(nn.Module):
    """Whisper encoder + trainable head. Trained on HC/PT depressed label (binary)."""  # radar_audio

    def __init__(self, freeze_encoder: bool = True) -> None:
        super().__init__()
        self.whisper = WhisperModel.from_pretrained(WHISPER_ID, low_cpu_mem_usage=True)
        if freeze_encoder:
            for param in self.whisper.parameters():
                param.requires_grad = False

        hidden_size = self.whisper.config.d_model
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),  # radar_audio: keep for binary depression; change for PHQ-8 regression
        )

    def forward(self, input_features: torch.Tensor) -> torch.Tensor:
        encoder_out = self.whisper.encoder(input_features)
        pooled = encoder_out.last_hidden_state.mean(dim=1)
        return self.head(pooled).squeeze(-1)


# --------------------------------------------------
# Dataset
# --------------------------------------------------

class AudioDepressionDataset(Dataset):
    def __init__(self, paths: list[str], labels: np.ndarray) -> None:
        self.paths = paths
        self.labels = labels.astype(np.float32)  # radar_audio: binary label or PHQ-8 score

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, float]:
        return load_audio(self.paths[idx]), float(self.labels[idx])


def whisper_collate(batch: list[tuple[np.ndarray, float]], proc: WhisperProcessor):
    waves, labels = zip(*batch)
    feats = proc(list(waves), sampling_rate=TARGET_SR, padding=True, return_tensors="pt")
    y = torch.tensor(labels, dtype=torch.float32)
    return feats.input_features, y


def train_one_epoch(model, loader, optimizer, loss_fn) -> float:
    model.train()
    total_loss = 0.0
    n_batches = 0
    for input_features, y in loader:
        input_features = input_features.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_features)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.inference_mode()
def predict_probs(model, loader) -> tuple[np.ndarray, np.ndarray]:
    # radar_audio: if using PHQ-8 regression, rename this to predict_values
    # and remove sigmoid.
    model.eval()
    all_probs: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []
    for input_features, y in loader:
        input_features = input_features.to(DEVICE)
        logits = model(input_features)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


# --------------------------------------------------
# Grouped train / test split (by participant_id)
# --------------------------------------------------

df = audio_df.reset_index(drop=True)

# radar_audio: confirm participant_id is from RADAR metadata, not filename parsing.
groups = df["participant_id"].astype(str)

# radar_audio: replace "depressed" if using PHQ-8 regression or another RADAR label column.
labels = df["depressed"].to_numpy(dtype=np.float32)

paths = df["file_path"].tolist()

# radar_audio: if RADAR already has official train/test split, use that instead of GroupShuffleSplit.
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, labels, groups=groups))

train_paths = [paths[i] for i in train_idx]
test_paths = [paths[i] for i in test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

train_loader = DataLoader(
    AudioDepressionDataset(train_paths, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=partial(whisper_collate, proc=processor),
)
test_loader = DataLoader(
    AudioDepressionDataset(test_paths, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=partial(whisper_collate, proc=processor),
)

print(
    f"Train: {len(train_paths)} rows ({df.iloc[train_idx]['participant_id'].nunique()} participants) | "
    f"Test: {len(test_paths)} rows ({df.iloc[test_idx]['participant_id'].nunique()} participants)"
)


# --------------------------------------------------
# Train + evaluate
# --------------------------------------------------

model = DAMLikeModel(freeze_encoder=FREEZE_ENCODER).to(DEVICE)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
)

# Binary classification loss.
# radar_audio: use BCEWithLogitsLoss only if RADAR target is binary depressed/not depressed.
# For PHQ-8 regression, use nn.MSELoss() or nn.SmoothL1Loss().
loss_fn = nn.BCEWithLogitsLoss()  # radar_audio

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    print(f"Epoch {epoch}/{EPOCHS} train_loss={train_loss:.4f}")

test_probs, test_true = predict_probs(model, test_loader)  # radar_audio
test_pred = (test_probs >= 0.5).astype(int)                # radar_audio

metrics = {
    "model": "DAM-like Whisper (HC/PT)",  # radar_audio: rename to DAM-like Whisper (RADAR)
    "n_total": len(df),
    "n_train": len(train_paths),
    "n_test": len(test_paths),
    "n_participants": df["participant_id"].nunique(),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "freeze_encoder": FREEZE_ENCODER,

    # radar_audio: classification metrics only if binary label.
    # For PHQ-8 regression use MAE, RMSE, R2, correlation, etc.
    "accuracy": accuracy_score(test_true, test_pred),
    "f1": f1_score(test_true, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(test_true, test_probs) if len(np.unique(test_true)) > 1 else np.nan,
}

summary_df = pd.DataFrame([metrics])

# Androids-specific columns.
# radar_audio: change selected columns based on RADAR metadata availability,
# e.g. task, language, site/hospital, phq8_score, split, recording_date.
pred_df = df.iloc[test_idx][
    ["file_path", "file", "file_stem", "subgroup", "participant_id", "depressed"]
].copy()  # radar_audio

pred_df["pred_prob"] = test_probs    # radar_audio
pred_df["pred_label"] = test_pred    # radar_audio

# Androids-specific filenames.
# radar_audio: rename outputs to dam_whisper_radar_summary.csv etc.
summary_path = RESULTS_PATH / "dam_whisper_hc_pt_summary.csv"  # radar_audio
pred_path = RESULTS_PATH / "dam_whisper_hc_pt_test_predictions.csv"  # radar_audio

summary_df.to_csv(summary_path, index=False)
pred_df.to_csv(pred_path, index=False)

print("\nHeld-out test metrics:")
print(summary_df.to_string(index=False))
print(f"\nSaved: {summary_path}")
print(f"Saved: {pred_path}")

# Quick smoke inference on the first training file
sample_path = train_paths[0]
sample_wav = load_audio(sample_path)
sample_inputs = processor(sample_wav, sampling_rate=TARGET_SR, return_tensors="pt")
with torch.inference_mode():
    sample_logit = model(sample_inputs.input_features.to(DEVICE))
    sample_prob = torch.sigmoid(sample_logit).item()  # radar_audio: remove sigmoid for regression

print(f"\nSmoke inference — {Path(sample_path).name}: P(depressed)={sample_prob:.3f}")  # radar_audio

model.cpu()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()